# A first tour of sonore

[sonore](https://github.com/choyun1/sonore) makes, manipulates and analyzes sounds the way hearing
scientists think about them. This notebook is a short tutorial you can run in the browser: on
[Google Colab](https://colab.research.google.com/github/choyun1/sonore/blob/main/docs/notebooks/start.ipynb)
it needs nothing but a Google account. Run the cells from top to bottom (Runtime > Run all). Every
sound comes with an audio player and four plots of it: its waveform, spectrum, spectrogram and
modulation spectrum. Headphones help, and are needed for the last section.

It goes in six steps:

1. [Sounds](#1.-Sounds): what a `Sound` is, levels, time and channels
2. [Canonical stimuli](#2.-Canonical-stimuli): tones, harmonic complexes, noise, modulation
3. [More complicated stimuli](#3.-More-complicated-stimuli): glides, ripples, iterated rippled
   noise, noise vocoding
4. [Sound textures](#4.-Sound-textures): new sounds from the statistics of a recording
5. [Voices](#5.-Voices): spectrograms, three F0 trackers, changing pitch and formants, a
   synthetic vowel
6. [Spatial hearing](#6.-Spatial-hearing): interaural cues, HRTFs, reverberation and movement

The [listening gallery](https://choyun1.github.io/sonore/gallery/) goes much further, with the code
for each example beside the sound it makes.

In [ ]:
%pip install --quiet sonore

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np

import sonore as so
from sonore import dB

fs = 44100
print("sonore", so.__version__)


def show(*sounds, fmax=None):
    """Plays each sound, with its overview below: waveform, spectrum, spectrogram and modulation
    spectrum."""
    for sound in sounds:
        display(sound)
        so.overview(sound, figsize=(8.4, 5.6), fmax=fmax)
        plt.show()


def download(path):
    """Downloads a file from the sonore repository on GitHub, once, and returns its local name."""
    filename = os.path.basename(path)
    if not os.path.exists(filename):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/choyun1/sonore/main/" + path, filename)
    return filename

## 1. Sounds

A `Sound` is an array of samples, one column per channel, and a sampling rate. Any NumPy array
becomes one.

In [ ]:
t = np.arange(int(0.5 * fs)) / fs
beep = so.Sound(0.1 * np.sin(2 * np.pi * 440 * t), fs)
print(beep.duration, beep.fs, beep.n_channels, beep.rms)
beep

Every generator returns a `Sound` with an RMS of 1, and anything stochastic takes `rng=` so it
can be reproduced exactly. A player plays a sound at its true level and can't go above full scale,
so the sounds here are set 20 dB down, to an RMS of 0.1. `show`, defined at the top, plays each
sound it is given and draws its overview below the player with `so.overview`.

Levels are written as levels: adding decibels changes a sound's level, and adding two sounds mixes
them. Times are in seconds, so `snd[0.1:0.4]` is the stretch from 100 to 400 ms, and `ramp` gives
a sound raised-cosine onsets and offsets.

In [ ]:
tone = so.pure_tone(0.5, fs, 1000).ramp(10e-3) - 20 * dB
noise = so.gaussian_noise(0.5, fs, rng=0).ramp(10e-3) - 20 * dB
tone_in_noise = tone + (noise + 5 * dB)  # the tone at -5 dB SNR
show(tone_in_noise, tone_in_noise[0.1:0.4])

`so.concat` plays sounds one after another and `so.silence` makes the gaps. A sound with two
channels is a left and a right ear; `so.Sound.from_channels` builds one, padding the shorter
channel with silence at the end.

In [ ]:
short = tone[0:0.2].ramp(10e-3)
steps = so.concat([short, so.silence(0.1, fs), short - 10 * dB, so.silence(0.1, fs), short - 20 * dB])
stereo = so.Sound.from_channels(steps, noise[0:0.25].ramp(10e-3))  # tones left, a noise burst right
show(steps, stereo)

## 2. Canonical stimuli

A harmonic complex is a sum of harmonics of one F0. Leave out the lowest harmonics, the
fundamental included, and you still hear its pitch (the missing fundamental). Schroeder phases flatten its envelope, so it sounds smoother
than the same harmonics in cosine phase, which add up to sharp pulses.

In [ ]:
cosine_phase = so.harmonic_complex(0.5, fs, f0=200, harmonics=range(1, 21)).ramp(10e-3) - 20 * dB
schroeder_phase = so.harmonic_complex(0.5, fs, f0=200, harmonics=range(1, 21), phases="schroeder+")
missing_fundamental = so.harmonic_complex(0.5, fs, f0=200, harmonics=range(3, 21))
show(cosine_phase, schroeder_phase.ramp(10e-3) - 20 * dB, missing_fundamental.ramp(10e-3) - 20 * dB)

Gaussian noise takes a band and a spectral tilt in dB per octave: 0 is white, -3 is pink.
`so.amplitude_modulate` multiplies a sound by a sinusoid; a tone modulated at 4 Hz has the carrier
and two sidebands 4 Hz either side, and its modulation spectrum has a peak at 4 Hz.

In [ ]:
pink = so.gaussian_noise(1, fs, band=(100, 8000), tilt=-3, rng=0).ramp(10e-3) - 20 * dB
modulated = so.amplitude_modulate(so.pure_tone(1, fs, 1000), f_mod=4, depth=1).ramp(10e-3) - 20 * dB
show(pink, modulated)

More on the [Classic stimuli](https://choyun1.github.io/sonore/gallery/classic.html) and
[Harmonics](https://choyun1.github.io/sonore/gallery/harmonics.html) pages.

## 3. More complicated stimuli

A glide sweeps a tone's frequency, linearly in hertz or exponentially (the same number of octaves
each second).

In [ ]:
linear_glide = so.linear_chirp(1, fs, 200, 4000).ramp(10e-3) - 20 * dB
exponential_glide = so.exponential_chirp(1, fs, 200, 4000).ramp(10e-3) - 20 * dB
show(linear_glide, exponential_glide)

A ripple is a sinusoidal pattern drawn on the time-frequency plane. This one drifts down at 4 Hz
with one cycle per octave, and you hear a sweep four times a second. The modulation spectrum, the
two-dimensional Fourier transform of the sound's envelopes, measures it back as a single peak.
More on the [Spectrotemporal ripples](https://choyun1.github.io/sonore/gallery/ripples.html) page.

In [ ]:
ripple = so.ripple_sound(so.Ripple(4, 1), 2, fs, rng=0).ramp(10e-3) - 20 * dB
show(ripple)
so.ModulationSpectrum.octave(ripple, f_lo=250, f_hi=8000).plot(wt_max=20, wf_max=4);

Iterated rippled noise adds noise to delayed copies of itself, again and again. The delay becomes
a pitch, 125 Hz for 8 ms, that grows clearer with each iteration although the sound stays a noise.
More on the [Iterated rippled noise](https://choyun1.github.io/sonore/gallery/irn.html) page.

In [ ]:
weak_pitch = so.iterated_ripple_noise(1, fs, delay=8e-3, iterations=1, rng=0).ramp(10e-3) - 20 * dB
strong_pitch = so.iterated_ripple_noise(1, fs, delay=8e-3, iterations=16, rng=0).ramp(10e-3) - 20 * dB
show(weak_pitch, strong_pitch, fmax=4000)

Noise vocoding keeps the slow envelope of each frequency band of a sound and replaces what is
inside it with noise. Speech stays understandable with as few as four bands (Shannon et al., 1995).

sonore does not ship recordings, so this cell downloads one from the sonore repository on GitHub:
"Providence had delivered him through the maelstrom", from the CMU ARCTIC corpus (Kominek & Black,
2004), whose licence allows this use. Its sources and licence are in
[docs/speech/SOURCES.md](https://github.com/choyun1/sonore/blob/main/docs/speech/SOURCES.md).
`so.load` reads any audio file that [soundfile](https://python-soundfile.readthedocs.io) can.
More on the [Noise vocoding](https://choyun1.github.io/sonore/gallery/vocoder.html) page.

In [ ]:
sentence = so.load(download("docs/speech/bdl_arctic_a0131.flac")).resample(fs)  # recorded at 16 kHz
four_bands = so.noise_vocode(sentence, n_bands=4, rng=0).normalize(rms=sentence.rms)
sixteen_bands = so.noise_vocode(sentence, n_bands=16, rng=0).normalize(rms=sentence.rms)
show(sentence, four_bands, sixteen_bands, fmax=5000)

## 4. Sound textures

Rain, fire and applause are made of many similar events, and what we recognize them by is their
statistics over time. sonore measures a texture's statistics through a model of the cochlea
(McDermott & Simoncelli, 2011), then starts from noise and adjusts it until its statistics match.
This cell measures 7 s of rain and synthesizes 2 s with 10 iterations, which takes about ten
seconds. The synthesis comes out at the model's own sampling rate, 20 kHz. The [Sound textures](https://choyun1.github.io/sonore/gallery/textures.html) page has
longer syntheses and explains each statistic.

In [ ]:
from sonore.texture import TextureStats
from sonore.texture.synth import synthesize

rain = so.load(download("docs/textures/rain.flac"))
statistics = TextureStats.measure(rain)
synthetic_rain, report = synthesize(statistics, duration=2.0, rng=1, max_iter=10)
print(f"The statistics match to {report['average_snr']:.0f} dB SNR on average.")
show(rain[0:2].ramp(10e-3).normalize(rms=0.1), synthetic_rain.ramp(10e-3).normalize(rms=0.1))

## 5. Voices

The window length decides what a spectrogram shows. A 5 ms window is shorter than one period of
this voice, so each glottal pulse stands out as a vertical stripe (a *wideband* spectrogram). A
33 ms window holds several periods, so the harmonics stand out as horizontal lines instead (a
*narrowband* one). The [Seeing speech](https://choyun1.github.io/sonore/gallery/speech.html) page
explains why, and what else a time-frequency picture can be.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.4, 2.8), layout="constrained")
for ax, win in zip(axes, [5e-3, 33e-3], strict=True):
    so.STFT(sentence, win_dur=win).plot(ax=ax, db_range=70, fmax=5000, colorbar=False)
    ax.set_title(f"{win * 1e3:.0f} ms window")

### Three ways to track F0

- The **cepstrum** is the Fourier transform of the log spectrum. The harmonics ripple the log
  spectrum once every F0 hertz, so the pitch shows up as a peak at one period.
- **`so.f0_track`** looks for the period in the waveform instead: it compares the sound with
  delayed copies of itself, as YIN does, and picks a smooth path through the candidates.
- **Harvest**, the tracker of the WORLD vocoder, is not part of sonore. The track below was
  computed with pyworld and stored next to the recording.

They agree where the voice is clear and part ways at the edges of voicing. The
[Cepstral analysis](https://choyun1.github.io/sonore/gallery/cepstrum.html) page compares them.

In [ ]:
cepstrum = so.Cepstrum(so.STFT(sentence, win_dur=0.040, hop_dur=0.005))
cepstral_t, cepstral_f0, _ = cepstrum.f0()
track = so.f0_track(sentence)
harvest_file = download("docs/speech/bdl_arctic_a0131_f0.csv")
harvest_t, harvest_f0 = np.loadtxt(harvest_file, delimiter=",", skiprows=2).T

fig, ax = plt.subplots(figsize=(8.4, 3.2), layout="constrained")
for times, f0, label, marker in [
    (harvest_t, harvest_f0, "Harvest (pyworld)", "x"),
    (track.t, track.f0[0], "so.f0_track", "+"),
    (cepstral_t, cepstral_f0[0], "cepstral peak", "."),
]:
    voiced = f0 > 0
    ax.plot(times[voiced], f0[voiced], marker, ms=4, label=label)
ax.set(xlabel="Time [s]", ylabel="F0 [Hz]", ylim=(50, 250))
ax.legend()
ax.grid(ls=":")

### Changing pitch and formants

WORLD-style analysis splits a voice into its F0, its spectral envelope (`so.cheaptrick`, the
formants) and its aperiodicity (`so.d4c`, how noisy each band is), and `so.world_synthesize` puts
them back together. Change one part before synthesis and only that part changes:
`so.scale_f0` raises the pitch, and `so.warp_frequency` stretches the envelope along frequency,
which moves the formants up as a shorter vocal tract would. More on the
[Aperiodicity](https://choyun1.github.io/sonore/gallery/aperiodicity.html) and
[Changing a voice](https://choyun1.github.io/sonore/gallery/voice.html) pages.

In [ ]:
harvest = (harvest_t, harvest_f0)
envelope = so.cheaptrick(sentence, harvest)
aperiodicity = so.d4c(sentence, harvest)

resynthesized = so.world_synthesize(harvest, envelope, aperiodicity, rng=0)
higher_pitch = so.world_synthesize(so.scale_f0(harvest, 1.5), envelope, aperiodicity, rng=0)
higher_formants = so.world_synthesize(harvest, so.warp_frequency(envelope, 1.2), aperiodicity, rng=0)
both = so.world_synthesize(so.scale_f0(harvest, 1.5), so.warp_frequency(envelope, 1.2), aperiodicity, rng=0)
voices = [resynthesized, higher_pitch, higher_formants, both]
show(*[voice.normalize(rms=sentence.rms) for voice in voices], fmax=5000)

A phase vocoder changes duration without changing pitch, and pitch without changing duration,
with no model of the voice at all; a pitch shift moves the formants too. The
[Phase vocoder](https://choyun1.github.io/sonore/gallery/pv.html) page shows how it works.

In [ ]:
slower = so.time_stretch(sentence, 1.5)  # same pitch, half again as long
lower = so.pitch_shift(sentence, -5)  # same duration, five semitones down
show(slower, lower, fmax=5000)

### A synthetic vowel

`so.klatt_synthesize` is a Klatt-style formant synthesizer: a glottal source through a cascade of
resonators, one per formant. These are the first three formants of the vowel in "hod" (Peterson
& Barney, 1952), with the pitch falling from 125 to 105 Hz. More on the
[Formant synthesis](https://choyun1.github.io/sonore/gallery/formants.html) page.

In [ ]:
vowel_duration = 1.0
vowel = so.klatt_synthesize(
    vowel_duration,
    fs,
    F0=([0, vowel_duration], [125, 105]),  # (times, values): a contour
    AV=([0, 0.02, vowel_duration - 0.05, vowel_duration], [0, 60, 60, 0]),  # voicing amplitude [dB]
    F1=730,
    F2=1090,
    F3=2440,
)
show(vowel.normalize(rms=0.05), fmax=5000)

## 6. Spatial hearing

Put on headphones. A 500 µs interaural time difference and a 6 dB level difference move noise to
the right, and `so.interaural_cues` measures both back in short windows. More on the
[Binaural cues](https://choyun1.github.io/sonore/gallery/binaural.html) page.

In [ ]:
lateral = so.apply_itd_ild(pink, itd=500e-6, ild=6)
show(lateral)
so.interaural_cues(lateral, win_dur=20e-3).plot();

Head-related transfer functions add the filtering of the head and ears, which carries elevation
and front-back as well. `so.load_hrirs()` downloads the PKU-IOA KEMAR responses measured 1 m away
(Qu et al., 2009; about 13 MB, once), and `so.spatialize` places a sound at a position in metres.
Here a noise burst steps from the left, through the front, to the right.

In [ ]:
hrirs = so.load_hrirs()
burst = pink[0:0.3].ramp(10e-3)
positions = [so.hcc_to_rect(100, 0, azimuth) for azimuth in (-90, -45, 0, 45, 90)]  # 1 m, degrees
placed = [so.spatialize(burst, position, hrirs) for position in positions]
gap = so.silence(0.1, fs, n_channels=2)
around = so.concat([part for sound in placed for part in (sound, gap)])
show(around)

A room adds reverberation. `so.synth_ir` builds a room impulse response from a reverberation time
(Traer & McDermott, 2016), and convolving a sound with it puts the sound in that room;
`so.measure_rt60` reads the reverberation time back, band by band. More on the
[Synthetic reverberation](https://choyun1.github.io/sonore/gallery/reverb.html) page.

In [ ]:
room = so.synth_ir(0.6, fs, rng=5)  # RT60 0.6 s
in_the_room = sentence.convolve(room).normalize(rms=sentence.rms)
show(in_the_room, fmax=5000)

band_centers, rt60s = so.measure_rt60(room)
fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
ax.semilogx(band_centers, rt60s, "o-")
ax.set(xlabel="Frequency [Hz]", ylabel="RT60 [s]", title="Reverberation time of the room")
ax.grid(ls=":")

Last, movement. `so.hcc_trajectory` describes a path by distance (cm), elevation and azimuth
(degrees), each a number, a `(times, values)` pair or a function of time, and `so.move_sound`
renders a sound along it. Here the talker walks around you 1.5 m away, from your left to your right,
first in free field and then in a room. More on the
[Moving talkers](https://choyun1.github.io/sonore/gallery/moving.html) page.

In [ ]:
walk_around = so.hcc_trajectory(150, 0, ([0, sentence.duration], [-80, 80]))
binaural_room = so.synth_ir(0.6, fs, n_channels=2, rng=1)  # a different tail at each ear
free_field = so.move_sound(sentence, walk_around, hrirs)
with_room = so.move_sound(sentence, walk_around, hrirs, room=binaural_room, drr_db=0.0)
quieter = sentence.rms / 2  # 6 dB down: the ear nearer the talker peaks high
show(free_field.normalize(rms=quieter), with_room.normalize(rms=quieter), fmax=5000)

## Where next

- The [listening gallery](https://choyun1.github.io/sonore/gallery/): textures, reverberation,
  moving talkers, vocoders, iterated rippled noise, classic stimuli, and more, each beside its
  code.
- The [README](https://github.com/choyun1/sonore#readme): conventions, and every module and what
  it holds.
- On your own machine: `pip install "sonore[notebook]"`.